# LangChain: LCEL Chains, RAG, and Memory

**Learning objectives:** Compose **LCEL** runnables (`|`, `RunnablePassthrough`, `RunnableLambda`), build a **RAG** chain with a mock retriever, use **Pydantic / JSON / Str** parsers, show **memory** patterns, and sketch **tool calling** with a stub model.

## Introduction

**LangChain** (with **langchain-core**) models pipelines as *runnables*. **LCEL** chains them with `|`. This notebook uses **`FakeListChatModel`** so nothing calls the network.

Install hints: `pip install langchain langchain-core pydantic` (optional `langchain` for classic `ConversationBufferMemory`).

In [ ]:
try:
    from langchain_core.language_models.fake import FakeListChatModel
    from langchain_core.output_parsers import StrOutputParser
    from langchain_core.prompts import ChatPromptTemplate
    from langchain_core.runnables import RunnableLambda, RunnableParallel, RunnablePassthrough
except ImportError:
    print("Install: pip install langchain-core langchain")
    raise

llm = FakeListChatModel(
    responses=[
        "Paris is the capital of France.",
        "RunnablePassthrough keeps keys while adding computed fields.",
    ]
)
prompt = ChatPromptTemplate.from_messages(
    [("system", "Answer in one short sentence."), ("human", "{q}")]
)
chain = prompt | llm | StrOutputParser()
print("Pipe:", chain.invoke({"q": "Capital of France?"}))

enrich = RunnablePassthrough.assign(n_chars=RunnableLambda(lambda d: len(d["text"])))
print("Passthrough.assign:", enrich.invoke({"text": "hello lcel"}))

para = RunnableParallel(
    upper=RunnableLambda(lambda d: d["t"].upper()),
    words=RunnableLambda(lambda d: len(d["t"].split())),
)
print("Parallel:", para.invoke({"t": "lang chain core"}))

### RAG chain with a mock retriever

Swap `mock_retriever` for vector search or hybrid retrieval in production.

In [ ]:
try:
    from langchain_core.documents import Document
    from langchain_core.language_models.fake import FakeListChatModel
    from langchain_core.output_parsers import StrOutputParser
    from langchain_core.prompts import ChatPromptTemplate
    from langchain_core.runnables import RunnableLambda, RunnablePassthrough
except ImportError:
    print("Install: pip install langchain-core")
    raise


def mock_retriever(query: str) -> list[Document]:
    return [
        Document(page_content="SLA: 99.9% monthly uptime for the API.", metadata={"source": "sla"}),
        Document(page_content="Incidents must be logged in the runbook.", metadata={"source": "runbook"}),
        Document(page_content=f"Echo: {query!r}", metadata={"source": "synthetic"}),
    ]


def format_docs(docs: list[Document]) -> str:
    return "\n\n".join(f"[{d.metadata.get('source')}] {d.page_content}" for d in docs)


retrieve = (
    RunnableLambda(lambda d: d["question"])
    | RunnableLambda(mock_retriever)
    | RunnableLambda(format_docs)
)
rag_prompt = ChatPromptTemplate.from_template(
    "Answer using only the context.\n\nContext:\n{context}\n\nQuestion: {question}"
)
llm_rag = FakeListChatModel(
    responses=["The SLA targets 99.9% monthly uptime for the API."]
)
rag_chain = RunnablePassthrough.assign(context=retrieve) | rag_prompt | llm_rag | StrOutputParser()
print(rag_chain.invoke({"question": "What uptime is mentioned?"}))

### Output parsers: Pydantic, JSON, Str

`StrOutputParser` returns plain text. **Pydantic** / **Json** parsers need model output that matches schema instructions (here the fake model returns canned JSON).

In [ ]:
try:
    from langchain_core.language_models.fake import FakeListChatModel
    from langchain_core.output_parsers import JsonOutputParser, PydanticOutputParser, StrOutputParser
    from langchain_core.prompts import ChatPromptTemplate
    from pydantic import BaseModel, Field
    from typing import Literal
except ImportError:
    print("Install: pip install langchain-core pydantic")
    raise


class Alert(BaseModel):
    code: str = Field(description="Short alert code")
    severity: Literal["low", "med", "high"] = Field(description="Severity bucket")


llm_s = FakeListChatModel(responses=["plain text ok"])
print("Str:", (ChatPromptTemplate.from_template("Hi {x}") | llm_s | StrOutputParser()).invoke({"x": "there"}))

parser_p = PydanticOutputParser(pydantic_object=Alert)
fmt = parser_p.get_format_instructions()
llm_p = FakeListChatModel(responses=['{"code": "CPU_SPIKE", "severity": "high"}'])
chain_p = (
    ChatPromptTemplate.from_template("Extract alert.\n{fmt}\nText: {text}\n").partial(fmt=fmt)
    | llm_p
    | parser_p
)
print("Pydantic:", chain_p.invoke({"text": "CPU_SPIKE is high severity."}))

parser_j = JsonOutputParser(pydantic_object=Alert)
fmt_j = parser_j.get_format_instructions()
llm_j = FakeListChatModel(responses=['{"code": "MEM", "severity": "low"}'])
chain_j = (
    ChatPromptTemplate.from_template("JSON only.\n{fmt}\n{text}\n").partial(fmt=fmt_j)
    | llm_j
    | parser_j
)
print("Json:", chain_j.invoke({"text": "memory soft limit"}))

### Memory: RunnableWithMessageHistory and ConversationBufferMemory

Prefer **`RunnableWithMessageHistory`** for LCEL. Classic **`ConversationBufferMemory`** still appears in older **`ConversationChain`** examples.

In [ ]:
try:
    from langchain_core.language_models.fake import FakeListChatModel
    from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
    from langchain_core.runnables.history import RunnableWithMessageHistory
    from langchain_core.chat_history import InMemoryChatMessageHistory
except ImportError:
    print("Install: pip install langchain-core")
    raise

llm_mem = FakeListChatModel(
    responses=["Got it.", "You said your favorite language is Python."]
)
prompt_h = ChatPromptTemplate.from_messages(
    [
        ("system", "Be concise."),
        MessagesPlaceholder("history"),
        ("human", "{input}"),
    ]
)
base = prompt_h | llm_mem
store: dict[str, InMemoryChatMessageHistory] = {}


def get_session_history(sid: str) -> InMemoryChatMessageHistory:
    if sid not in store:
        store[sid] = InMemoryChatMessageHistory()
    return store[sid]


with_history = RunnableWithMessageHistory(
    base,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history",
)
cfg = {"configurable": {"session_id": "demo-user"}}
print("T1:", with_history.invoke({"input": "My favorite language is Python."}, config=cfg))
print("T2:", with_history.invoke({"input": "What is my favorite language?"}, config=cfg))

try:
    from langchain.chains import ConversationChain
    from langchain.memory import ConversationBufferMemory
except ImportError:
    print("Optional: pip install langchain for ConversationBufferMemory demo")
else:
    mem = ConversationBufferMemory(return_messages=True)
    classic = ConversationChain(llm=llm_mem, memory=mem, verbose=False)
    print("Classic:", classic.invoke({"input": "Say hi in one word."}))

### Tool calling patterns

`@tool` defines schemas. **`FakeListChatModel`** may not emit real `tool_calls`; we show **bind_tools** when it works and a **manual** `ToolMessage` round-trip.

In [ ]:
from typing import Any

try:
    from langchain_core.language_models.fake import FakeListChatModel
    from langchain_core.messages import AIMessage, HumanMessage, ToolMessage
    from langchain_core.tools import tool
except ImportError:
    print("Install: pip install langchain-core")
    raise


@tool
def multiply(a: float, b: float) -> float:
    # Return a * b (docstring avoided for notebook embedding)
    return a * b


tools = [multiply]
print("Tool fields:", list(multiply.args_schema.model_json_schema().get("properties", {}).keys()))

llm_t = FakeListChatModel(responses=["I'll call multiply."])
try:
    msg = llm_t.bind_tools(tools).invoke("What is 6 * 7?")
    print("bind_tools:", msg)
except Exception as exc:
    print("(bind_tools may be limited for FakeListChatModel):", exc)

fake_call = {"name": "multiply", "args": {"a": 6, "b": 7}, "id": "tc1", "type": "tool_call"}
ai = AIMessage(content="", tool_calls=[fake_call])
human = HumanMessage(content="Use multiply for 6*7.")
result = multiply.invoke(fake_call["args"])
tool_msg = ToolMessage(content=str(result), tool_call_id=fake_call["id"])
print("Human:", human.content)
print("AI tool_calls:", ai.tool_calls)
print("ToolMessage:", tool_msg.content)

## Conclusion

- **LCEL** (`|`, `RunnablePassthrough`, `RunnableLambda`, `RunnableParallel`) keeps retrieval, prompting, and parsing composable.
- **RAG** is retrieve, format, prompt, model, parse; swap in real search when ready.
- **Parsers** bridge free-form LLM text to typed app data.
- **Memory:** prefer **`RunnableWithMessageHistory`**; know **`ConversationBufferMemory`** for legacy stacks.
- **Tools:** `@tool`, bind, execute, append **`ToolMessage`**, then continue the conversation.

Swap **`FakeListChatModel`** for a real chat model when you leave local experiments.